### Data preprocessing


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
BASE = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/"
full = pd.read_csv(BASE + "training_data.csv")
test = pd.read_csv(BASE + "testing_data.csv") # scellé jusqu'à la Partie F
full["id"], test["id"] = range(len(full)), range(len(test))


In [13]:
print(full.isnull().sum())

tweet        0
sarcasm      0
sentiment    0
dialect      0
id           0
dtype: int64


### full Dups, tweet Dups

In [14]:
full._dups = full.duplicated()
print(f"Number of duplicate lines in the full set: {full._dups.sum()}")

Number of duplicate lines in the full set: 0


C:\Users\user\AppData\Local\Temp\ipykernel_29408\175053858.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  full._dups = full.duplicated()


In [15]:
tweet_dups = full["tweet"].duplicated(keep="first")
print(f"Number of duplicate tweets in the training set: {tweet_dups.sum()}")


Number of duplicate tweets in the training set: 61


In [16]:
dup_tweets = []

def show_dup_tweets(df):
    for index, row in df.iterrows():
        if df.duplicated(subset=["tweet"], keep=False).iloc[index]:
            dup_tweets.append(row)

show_dup_tweets(full)
dup_tweets_df = pd.DataFrame(dup_tweets)
display(dup_tweets_df)

,tweet,sarcasm,sentiment,dialect,id
139,"""RT @mh_awadi: #مصر في الأخبار :أغذية ملوثةأكل...",True,NEG,egypt,139
163,"""RT @A822822A: ياروافض بلا كلام وإلا سوف أخرج♦...",True,NEG,msa,163
541,"""RT @safa_tv: #العبادي يرد على #أردوغان :بالتأ...",False,NEG,msa,541
646,"""RT @BassamJaara: حلب لن تسقط واهلها لن يغادرو...",False,NEG,msa,646
689,"""RT @SYRIA_KIDS: شكراً #بشارآثرتهم على نفسك .و...",False,NEG,msa,689
...,...,...,...,...,...
11872,"""@sawabcenter داعش مجرمين قتلة ولكن النظام الس...",True,NEG,msa,11872
11897,"""RT @HUN_0_TER: @Counselkremlin الرئيس #الروسي...",False,NEU,msa,11897
12158,"""RT @ajmubasher: وزير خارجية #بريطانيا يدعو لل...",False,NEU,msa,12158
12280,"""RT @M__albugaily: هاجموا الحبيبة مصر بضراوة ل...",False,NEG,egypt,12280


In [17]:
display(dup_tweets_df)

,tweet,sarcasm,sentiment,dialect,id
139,"""RT @mh_awadi: #مصر في الأخبار :أغذية ملوثةأكل...",True,NEG,egypt,139
163,"""RT @A822822A: ياروافض بلا كلام وإلا سوف أخرج♦...",True,NEG,msa,163
541,"""RT @safa_tv: #العبادي يرد على #أردوغان :بالتأ...",False,NEG,msa,541
646,"""RT @BassamJaara: حلب لن تسقط واهلها لن يغادرو...",False,NEG,msa,646
689,"""RT @SYRIA_KIDS: شكراً #بشارآثرتهم على نفسك .و...",False,NEG,msa,689
...,...,...,...,...,...
11872,"""@sawabcenter داعش مجرمين قتلة ولكن النظام الس...",True,NEG,msa,11872
11897,"""RT @HUN_0_TER: @Counselkremlin الرئيس #الروسي...",False,NEU,msa,11897
12158,"""RT @ajmubasher: وزير خارجية #بريطانيا يدعو لل...",False,NEU,msa,12158
12280,"""RT @M__albugaily: هاجموا الحبيبة مصر بضراوة ل...",False,NEG,egypt,12280


In [18]:
full.drop_duplicates(subset=["tweet"], keep=False, inplace=True)

In [19]:
full["tweet"].isnull().sum()

np.int64(0)

### Mapping Emojis in arabic


In [20]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4
5,"""طبيب #المقاصة: إصابة #أحمد_الشيخ بسيطة تحتاج ...",False,NEG,msa,5
6,مرسي : مش هنام اكتر من اربع ساعات في اليوم.......,True,NEG,egypt,6
7,#انتخبوا_البرص مشكلة,True,NEG,egypt,7
8,"""Retweeted هاشتاق العرب (@TheArabHash):#فيديو ...",False,POS,msa,8
9,▪️محافظ العاصمة تفقد سير العملية الانتخابية في...,False,NEU,msa,9


In [21]:
from arabic_emojipedia.emoji_description import get_emoji_description
import emoji

emoji = "😊"

description = get_emoji_description(emoji)

print(f"Description for {emoji}: {description}")

Description for 😊: وجه مبتسم بعيون مبتسمة


In [24]:
import emoji
from arabic_emojipedia.emoji_description import get_emoji_description

def replace_emoji_with_desc(text):
    if not isinstance(text, str):
        return text
    return emoji.replace_emoji(text, replace=lambda chars: f" {get_emoji_description(chars)} ")

full["tweet"] = full["tweet"].apply(replace_emoji_with_desc)

TypeError: replace_emoji_with_desc.<locals>.<lambda>() takes 1 positional argument but 2 were given

In [ ]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار وجه مبتسم بعيون قلب ...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4
5,"""طبيب #المقاصة: إصابة #أحمد_الشيخ بسيطة تحتاج ...",False,NEG,msa,5
6,مرسي : مش هنام اكتر من اربع ساعات في اليوم.......,True,NEG,egypt,6
7,#انتخبوا_البرص مشكلة,True,NEG,egypt,7
8,"""Retweeted هاشتاق العرب (@TheArabHash):#فيديو ...",False,POS,msa,8
9,مربع صغير أسود محافظ العاصمة تفقد سير العملية...,False,NEU,msa,9


### Punctuation and special characters removal

Keeping the @ and # actual values as they can be important

In [ ]:
full.head

<bound method NDFrame.head of                                                    tweet  sarcasm sentiment  \
0      "د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...    False       NEU   
1      "مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb...    False       NEU   
2      “الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...     True       NEG   
3      "@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...     True       NEG   
4      "قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...    False       NEU   
...                                                  ...      ...       ...   
12542  #المصري_اليوم #علاء_عبدالفتاح: #مصر لم تشهد حك...    False       NEG   
12543  هلأ صاير انت يا #فلعوط بدك تعطي محاضرات ع تويت...     True       NEG   
12544  "لا اله الا الله💜#أيفون_البروفيسور https://t.c...    False       NEU   
12546  RT @Yousiif65: هاري بوتر؟👓🎩 https://t.co/959Oo...    False       POS   
12547        النت يلعن شكله طول الوقت تمام لين بدا الحفل     True       NEG   

      dialect     id 

In [ ]:
import re

def remove_punctuation(text):
    text = re.sub(r'[^\w\s ! ?]','', text)
    text = re.sub(r'@','', text)
    text = re.sub(r'#','', text)
    text = re.sub(r'http\S+','', text)
    text = re.sub(r'www\.\S+','', text)
    text = re.sub(r'_',' ', text)
    return text

full['tweet'] = full['tweet'].apply(remove_punctuation)

full['tweet'].head(10)

0    د محمود العلايليأرى أن الفريق أحمد شفيق رقم مه...
1                            مع فيدرر يا آجا والكبار  
2    الداعون لمبدأ الاختلاط بين الجنسين كالداعين لإ...
3    ihe 94 ya78m amooo5 badiajnikhar Oukasafa reos...
4    قل شرق حلب ولا تقل حلب الشرقية وقل غرب حلب ولا...
5    طبيب المقاصة إصابة أحمد الشيخ بسيطة تحتاج للرا...
6    مرسي  مش هنام اكتر من اربع ساعات في اليوم لو ن...
7                                  انتخبوا البرص مشكلة
8    Retweeted هاشتاق العرب TheArabHashفيديو  أهالي...
9    محافظ العاصمة تفقد سير العملية الانتخابية في ع...
Name: tweet, dtype: str

### Translation

In [ ]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار وجه مبتسم بعيون قلب ...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4
5,"""طبيب #المقاصة: إصابة #أحمد_الشيخ بسيطة تحتاج ...",False,NEG,msa,5
6,مرسي : مش هنام اكتر من اربع ساعات في اليوم.......,True,NEG,egypt,6
7,#انتخبوا_البرص مشكلة,True,NEG,egypt,7
8,"""Retweeted هاشتاق العرب (@TheArabHash):#فيديو ...",False,POS,msa,8
9,مربع صغير أسود محافظ العاصمة تفقد سير العملية...,False,NEU,msa,9


In [ ]:
from googletrans import Translator
import re 

translator = Translator()

def translate_text(text):
    latin_words = re.findall(r'[a-zA-Z0-9]+', text)

    for la in latin_words:
        if la in text:
            try:
                translated_word = translator.translate(la, src='en', dest='ar').text
                text = re.sub(rf'\b{re.escape(la)}\b', translated_word, text)
            except Exception:
                continue
    return text

full['tweet'] = full['tweet'].apply(translate_text)

TypeError: expected string or bytes-like object, got 'coroutine'

In [ ]:
full.head(10)

,tweet,sarcasm,sentiment,dialect,id
0,<coroutine object Translator.translate at 0x00...,False,NEU,msa,0
1,<coroutine object Translator.translate at 0x00...,False,NEU,msa,1
2,<coroutine object Translator.translate at 0x00...,True,NEG,msa,2
3,<coroutine object Translator.translate at 0x00...,True,NEG,gulf,3
4,<coroutine object Translator.translate at 0x00...,False,NEU,msa,4
5,<coroutine object Translator.translate at 0x00...,False,NEG,msa,5
6,<coroutine object Translator.translate at 0x00...,True,NEG,egypt,6
7,<coroutine object Translator.translate at 0x00...,True,NEG,egypt,7
8,<coroutine object Translator.translate at 0x00...,False,POS,msa,8
9,<coroutine object Translator.translate at 0x00...,False,NEU,msa,9


### Data split

In [ ]:
train, val = train_test_split(full, test_size=0.10, random_state=42,
stratify=full["sarcasm"])

### Cleaned data export

In [ ]:
from IPython import paths
output_dir = Path(r"C:\ghoufrane\Master\M2\llms\Hanwritten-code-For-LLM-FineTuning\Data\cleaned_splits")
output_file_train = Path(f"{output_dir}\processed_train.csv")
output_file_val = Path(f"{output_dir}\processed_val.csv")

train 

train.drop(columns=["tweet"]).to_csv(output_file_train,index=False)
val.drop(columns=["tweet"]).to_csv(output_file_val, index=False)